# Open Source Text Embeddings

In [1]:
from utils.llm import get_embedding_model

embed_model = get_embedding_model(model="BAAI/bge-m3")

/raid/p-storage/joshuamihai_e64ef832/.local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [8]:
from utils.utils import load_from_pickle

all_documents = load_from_pickle("all_documents")

In [3]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    all_documents, embed_model, collection_name="uu-all", path="qdrant"
)

KeyboardInterrupt: 

In [9]:
retriever = vectorstore.as_retriever()
retriever.invoke("What are the provinces of South Sumatera")

[Document(metadata={'title': 'Undang-undang Nomor 17 Tahun 2022 Tentang Provinsi Sumatera Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 17, 'tahun': 2022, 'tentang': 'Provinsi Sumatera Barat', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '25 Juli 2022', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu17-2022.pdf', 'id': 'cayt2uju', 'filename': 'uu_25_juli_2022_uu17_2022_cayt2uju.pdf', 'page_number': 3, '_id': 'a7b905de8c204d558a3329182dc43d57', '_collection_name': 'uu'}, page_content='DOC NAME: Undang-undang Nomor 17 Tahun 2022 Tentang Provinsi Sumatera Barat | UNDANG-UNDANG\n\n\n- 3 -\nBAB II\nCAKUPAN WILAYAH, IBU KOTA, DAN KARAKTERISTIK\nPROVINSI SUMATERA BARAT\nPasal 3\n(1) Provinsi Sumatera Barat terdiri atas 12 (dua belas)\nkabupaten dan 7 (tujuh) kota, yaitu:\na. Kabupaten Pesisir Selatan;\nb. Kabupaten Solok;.\nc. Kabupaten Sijunjung;\nd. Kabupaten Tanah Datar;\ne. Kabupaten Padang Pariaman;\nf. K

In [13]:
from langchain_community.chat_models import ChatOllama

llm =ChatOllama(model="llama3.1", temperature=0.1)
llm.invoke("What are the provinces of South Sumatera")

AIMessage(content='The province of South Sumatra (also known as Sumatera Selatan) in Indonesia is divided into 12 regencies and 2 cities. Here are the names of the provinces:\n\n1. Banyuasin Regency\n2. Lahat Regency\n3. Musi Banyuasin Regency\n4. Musi Rawas Regency\n5. Ogan Ilir Regency\n6. Ogan Komering Ulu Regency\n7. Ogan Komering Ulu Selatan Regency\n8. Prabumulih City\n9. Penukal Abung Regency\n10. Pesawaran Regency\n11. South Tengah Regency\n12. Tanjung Jabung Barat Regency\n13. Tanjung Jabung Timur Regency', response_metadata={'model': 'llama3.1', 'created_at': '2024-08-05T15:41:36.8208801Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 3123565000, 'load_duration': 14573500, 'prompt_eval_count': 19, 'prompt_eval_duration': 241298000, 'eval_count': 167, 'eval_duration': 2866674000}, id='run-a5fd3984-16d5-4e12-8aa4-c9d89a677980-0')

# Trying out QdrantIndexManager

In [6]:
def clean_document_content(content):
    return content.replace('\x00', '')  # Remove NUL characters

In [7]:
from utils.llm import get_embedding_model
from utils.utils import load_from_pickle


embed_model = get_embedding_model(model="BAAI/bge-m3")
all_documents = load_from_pickle("all_documents")

In [9]:
from utils.document.split import document_splitter

all_splits = document_splitter(docs=all_documents)
all_splits_sorted = sorted(all_splits, key=lambda x: (x.metadata['id'], x.metadata.get('page_number', '0')))
for split in all_splits_sorted:
    split.page_content = clean_document_content(split.page_content)

Split 46409 documents into 126966 chunks


In [10]:
from utils.database.qdrant import QdrantIndexManager

qdrant_index_manager = QdrantIndexManager(embed_model, index_name="uu-all")

In [11]:
qdrant_index_manager.store_vector_index(all_splits, batch_size=5000)

Loaded 1-5000 documents
Loaded 5001-10000 documents
Loaded 10001-15000 documents
Loaded 15001-20000 documents
Loaded 20001-25000 documents
Loaded 25001-30000 documents
Loaded 30001-35000 documents
Loaded 35001-40000 documents
Loaded 40001-45000 documents
Loaded 45001-50000 documents
Loaded 50001-55000 documents
Loaded 55001-60000 documents
Loaded 60001-65000 documents
Loaded 65001-70000 documents
Loaded 70001-75000 documents
Loaded 75001-80000 documents
Loaded 80001-85000 documents
Loaded 85001-90000 documents
Loaded 90001-95000 documents
Loaded 95001-100000 documents
Loaded 100001-105000 documents
Loaded 105001-110000 documents
Loaded 110001-115000 documents
Loaded 115001-120000 documents
Loaded 120001-125000 documents
Loaded 125001-126966 documents


In [12]:
vector_store = qdrant_index_manager.load_vector_index()

Loaded index 'uu-all'.


In [16]:
if vector_store:
    results = vector_store.similarity_search("Apa itu capital Jawa Barat?", k=50)
    for result in results:
        print(result.page_content, result.metadata)

NEPUEUT INDONE;IA
-4-
t. Kota Sukabumi;
u. Kota Bandung;
v. Kota Cirebon;
w. Kota Bekasi;
x. Kota Depok;
y. Kota Cimahi;
z. Kota Tasikmalaya; dan
aa. Kota Banjar.
Pasal 4
Ibu kota Provinsi Jawa Barat berkedudukan di Kota
Bandung.
Pasal 5
Provinsi Jawa Barat memiliki karakteristik, yaitu:
a.
dengan ciri geogralis utama kawasan
dataran rendah berupa pesisir dan pantai, kawasan
dataran tinggi berupa pegunungan dan
kawasan taman nasional serta kawasan lindung dan
konservasi yang merupakan bagian dari potensi
kewilayahan Provinsi Jawa Barat;
b. potensi sumber daya alam berupa pertanian,
kehutanan, kelautan dan perikanan, pertambangan,
potensi pariwisata, potensi perdagangan, potensi
industri, potensi ekonomi kreatif, dan potensi lainnya;
dan
c. suku bangsa dan kultural yang secara umum memiliki
karakter religius dan berbudaya sekaligus menjunjung
tinegi adat istiadat dan kelestarian lingkungan.
BABIII ...
SK No l8ll04A

Source: [Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat]

In [25]:
# qdrant_index_manager.delete_index()

Deleted index 'uu'.
